## AUTHOR     : UZAIR-UR-REHMAN
## REG ID	  : 22JZELE0471
## DEPARTMENT : ELECTRICAL
## EMAIL ID   : uzairkhattakh@gmail.com

## Lab 11: 1D CNN for Time Series Forecasting

### Import Libraries

In [1]:
import os
os.chdir(r'D:\UNIVERSITY\8 SEMESTER\MACHINE LEARNING LAB\LAB 10')

In [2]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

### Model Parameters

In [3]:
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

### Define 1D CNN Model

In [4]:
def CNN():
    input_data = Input(shape=(time_steps, num_features))
    x1 = Conv1D(16, 2, activation="relu")(input_data)
    x2 = Conv1D(16, 2, activation="relu")(x1)
    flatten = Flatten()(x2)
    output_data = Dense(1)(flatten)
    model = Model(input_data, output_data)
    return model

### Model Summary

In [5]:
model1 = CNN()
model1.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 24, 21)]          0         
                                                                 
 conv1d (Conv1D)             (None, 23, 16)            688       
                                                                 
 conv1d_1 (Conv1D)           (None, 22, 16)            528       
                                                                 
 flatten (Flatten)           (None, 352)               0         
                                                                 
 dense (Dense)               (None, 1)                 353       
                                                                 
Total params: 1,569
Trainable params: 1,569
Non-trainable params: 0
_________________________________________________________________


In [6]:
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) and install graphviz (see instructions at https://graphviz.gitlab.io/download/) for plot_model to work.


### Callbacks Setup

In [7]:
checkpoints = r'D:\UNIVERSITY\8 SEMESTER\MACHINE LEARNING LAB\LAB 10\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'D:\UNIVERSITY\8 SEMESTER\MACHINE LEARNING LAB\LAB 10'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

In [8]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]

In [9]:
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model =CNN()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


### Load and Prepare Dataset

In [10]:
import os
path_dataset =r'D:\UNIVERSITY\8 SEMESTER\MACHINE LEARNING LAB\LAB 10'
path_tr = os.path.join(path_dataset, 'train.csv')
df_tr = pd.read_csv(path_tr)
train_set = df_tr.iloc[:].values
path_v = os.path.join(path_dataset, 'validation.csv')
df_v = pd.read_csv(path_v)
validation_set = df_v.iloc[:].values 
path_te = os.path.join(path_dataset, 'test.csv')
df_te = pd.read_csv(path_te)
test_set = df_te.iloc[:].values 

path_scaler = os.path.join(path_dataset, 'AEP_scaler.pkl')
scaler         = pickle.load(open(path_scaler, 'rb'))

train_set.shape, validation_set.shape, test_set.shape

c:\Users\PMLS\anaconda3\envs\ML\lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


((860, 21), (90, 21), (30, 21))

In [11]:
time_steps=24
num_features=21

### Data Splitting (Univariate Multi-step)

In [12]:
start = time.time()
train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0,target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0,target_len=1)
print('Time Consumed', time.time()-start, "sec")

Time Consumed 0.0 sec


### Model Training - Phase 1

In [13]:
epochs = 60
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,verbose = verbose)

Epoch 1/60
 1/27 [>.............................] - ETA: 8s - loss: 1.3341 - mae: 1.3341 - mape: 669.0480
Epoch 1: val_loss improved from inf to 0.15548, saving model to D:\UNIVERSITY\8 SEMESTER\MACHINE LEARNING LAB\LAB 10\E1-cp-0001-loss0.16.h5
27/27 [==============================] - 1s 22ms/step - loss: 0.5027 - mae: 0.5027 - mape: 222.4852 - val_loss: 0.1555 - val_mae: 0.1555 - val_mape: 63.3388
Epoch 2/60
 1/27 [>.............................] - ETA: 0s - loss: 0.1658 - mae: 0.1658 - mape: 163.0910
Epoch 2: val_loss improved from 0.15548 to 0.09770, saving model to D:\UNIVERSITY\8 SEMESTER\MACHINE LEARNING LAB\LAB 10\E1-cp-0002-loss0.10.h5
27/27 [==============================] - 0s 10ms/step - loss: 0.1114 - mae: 0.1114 - mape: 65.9026 - val_loss: 0.0977 - val_mae: 0.0977 - val_mape: 29.7893
Epoch 3/60
12/27 [============>.................] - ETA: 0s - loss: 0.0780 - mae: 0.0780 - mape: 32.8629
Epoch 3: val_loss improved from 0.09770 to 0.06083, saving model to D:\UNIVERSITY\8 SE

### Model Evaluation - Phase 1

In [14]:

model = load_model(r'D:\UNIVERSITY\8 SEMESTER\MACHINE LEARNING LAB\LAB 10\E1-cp-0050-loss0.03.h5')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 [==============================] - 0s 54ms/step
Mean Absolute Error (MAE): 2714.3
Median Absolute Error (MedAE): 2770.24
Mean Squared Error (MSE): 7399674.15
Root Mean Squared Error (RMSE): 2720.23
Mean Absolute Percentage Error (MAPE): 17.33 %
Median Absolute Percentage Error (MDAPE): 17.95 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


### Model   (Fine-tuning)

In [15]:
checkpoints = r'D:\UNIVERSITY\8 SEMESTER\MACHINE LEARNING LAB\LAB 10'
model=r'D:\UNIVERSITY\8 SEMESTER\MACHINE LEARNING LAB\LAB 10\E1-cp-0029-loss0.02.h5'
start_epoch= 58

In [16]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] loading D:\UNIVERSITY\8 SEMESTER\MACHINE LEARNING LAB\LAB 10\E1-cp-0029-loss0.02.h5...
[INFO] old learning rate: 0.0010000000474974513
[INFO] new learning rate: 9.999999747378752e-05


 ### Model Training  (Fine-tuning)

In [17]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/10
 1/27 [>.............................] - ETA: 3s - loss: 0.0229 - mae: 0.0229 - mape: 6.3792
Epoch 1: val_loss improved from inf to 0.02501, saving model to D:\UNIVERSITY\8 SEMESTER\MACHINE LEARNING LAB\LAB 10


INFO:tensorflow:Assets written to: D:\UNIVERSITY\8 SEMESTER\MACHINE LEARNING LAB\LAB 10\assets


INFO:tensorflow:Assets written to: D:\UNIVERSITY\8 SEMESTER\MACHINE LEARNING LAB\LAB 10\assets


27/27 [==============================] - 1s 25ms/step - loss: 0.0192 - mae: 0.0192 - mape: 8.2943 - val_loss: 0.0250 - val_mae: 0.0250 - val_mape: 6.9807
Epoch 2/10
 1/27 [>.............................] - ETA: 0s - loss: 0.0191 - mae: 0.0191 - mape: 14.1339
Epoch 2: val_loss did not improve from 0.02501
27/27 [==============================] - 0s 5ms/step - loss: 0.0177 - mae: 0.0177 - mape: 7.5537 - val_loss: 0.0267 - val_mae: 0.0267 - val_mape: 7.7074
Epoch 3/10
 1/27 [>.............................] - ETA: 0s - loss: 0.0216 - mae: 0.0216 - mape: 6.8626
Epoch 3: val_loss did not improve from 0.02501
27/27 [==============================] - 0s 5ms/step - loss: 0.0177 - mae: 0.0177 - mape: 7.6135 - val_loss: 0.0268 - val_mae: 0.0268 - val_mape: 7.7427
Epoch 4/10
 1/27 [>.............................] - ETA: 0s - loss: 0.0140 - mae: 0.0140 - mape: 10.9037
Epoch 4: val_loss did not improve from 0.02501
27/27 [==============================] - 0s 7ms/step - loss: 0.0172 - mae: 0.0172 - m

### Model Evaluation - Phase 2

In [18]:

model = load_model(r'D:\UNIVERSITY\8 SEMESTER\MACHINE LEARNING LAB\LAB 10\E1-cp-0029-loss0.02.h5')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 [==============================] - 0s 57ms/step
Mean Absolute Error (MAE): 19710.06
Median Absolute Error (MedAE): 19992.61
Mean Squared Error (MSE): 388722857.94
Root Mean Squared Error (RMSE): 19716.06
Mean Absolute Percentage Error (MAPE): 125.95 %
Median Absolute Percentage Error (MDAPE): 127.2 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)
